# STORM-PhysNet Master Reproduction Notebook
This notebook provides a complete walk-through of the STORM-PhysNet paper results.

## 1. Setup
Clone repository and setup sys.path

In [ ]:
import os, sys
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/bnsama29-cloud/STORM-PhysNet.git"
REPO_DIR = Path("STORM-PhysNet")

if not REPO_DIR.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))
    os.chdir(REPO_DIR)

## 2. Official Tables
Load pre-computed aggregated tables from `results/` to confirm exact paper metrics.

In [ ]:
import pandas as pd
import json

means = pd.read_csv("results/table_main_means.csv", index_col=0)
print("=== Main PE Means ===")
print(means[["PE_1h", "PE_6h", "PE_12h"]].round(3))

# Assert paper numbers approximately hold
storm_1h = means.loc["storm_bz", "PE_1h"]
storm_6h = means.loc["storm_bz", "PE_6h"]
print(f"\nSTORM-Bz 1h PE: {storm_1h:.3f} (Paper: ~0.986)")
print(f"STORM-Bz 6h PE: {storm_6h:.3f} (Paper: ~0.900)")

## 3. Data Pipeline
Load the GOES and OMNI datasets and create dataloaders.

In [ ]:
import yaml
from src.data.cdf_reader import read_goes_directory, read_wind_directory
from src.data.preprocessor import Preprocessor
from src.data.dataloader import make_dataloaders

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

goes_df = read_goes_directory("datasets/goes")
wind_df = read_wind_directory("datasets/omni")
raw_df = goes_df.join(wind_df, how="inner")

preprocessor = Preprocessor()
train_df, val_df, test_df = preprocessor.fit_transform(raw_df)

seq_len = int(cfg["data"]["sequence_length"])
train_loader, val_loader, test_loader = make_dataloaders(
    train_df, val_df, test_df, seq_len=seq_len, batch_size=64
)
print(f"Train/Val/Test windows: {len(train_loader.dataset)} / {len(val_loader.dataset)} / {len(test_loader.dataset)}")

## 4. Optional Demo Train
Run a single 1-epoch training loop to verify the API.

In [ ]:
import torch
from src.training.trainer import Trainer

# Optional short demonstration (set DEMO_TRAIN=True)
DEMO_TRAIN = False

if DEMO_TRAIN:
    demo_cfg = deepcopy(cfg)
    demo_cfg["training"]["epochs"] = 1
    
    n_sw = int(next(iter(train_loader))["x_sw"].shape[-1])
    trainer = Trainer(demo_cfg)
    model = trainer.build_model(n_sw)
    
    print("Running 1-epoch demo training...")
    # NOTE: Our Trainer API uses .train() internally, returning val_loss.
    val_loss = trainer.train(model, train_loader, val_loader, device=torch.device("cpu"))
    print(f"Demo complete. Val Loss: {val_loss:.4f}")

## 5. Optional Eval
Load a pre-trained checkpoint and run evaluation.

In [ ]:
# Optional evaluation
# Requires checkpoints to be available locally in checkpoints/storm_bz/seed_42/
from src.evaluation.metrics import prediction_efficiency

DEMO_EVAL = False

if DEMO_EVAL:
    ckpt_path = Path("checkpoints/storm_bz/seed_42/storm_physnet_bz_best.pt")
    if ckpt_path.exists():
        trainer = Trainer(cfg)
        model = trainer.build_model(n_sw)
        model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=True))
        model.eval()
        # [Add standard forward pass logic here...]
        print("Model loaded successfully.")
    else:
        print("Checkpoint not found for demo eval.")

## 6. GRASP Domain Transfer
Review fine-tuning transfer capability on Indian-longitude satellite data.

In [ ]:
grasp_results = pd.read_csv("results/table_grasp_storm_bz.csv")
print("=== GRASP Domain Transfer (Zero-shot vs Fine-tuned) ===")
print(grasp_results)

## 7. Figure Plotting
Generate horizon plots from the official mean tables.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plot_models = ["lstm", "transformer", "transformer_matched", "storm_bz"]
subset = means.loc[plot_models]

for col, marker in zip(["PE_1h", "PE_6h", "PE_12h"], ["o", "s", "^"]):
    plt.plot(plot_models, subset[col], marker=marker, label=col)

plt.title("Prediction Efficiency by Horizon")
plt.ylabel("PE")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Reviewer Map
Summary of which files back which claims:
- **`table_main_means.csv`**: Core paper evaluation table.
- **`table_bagged.csv`**: True bagging validation.
- **`ablation_final_table.csv`**: Physics-informed component validation.
- **`table_grasp_storm_bz.csv`**: Fine-tuning domain shift transfer.
- **`ensemble_summary.json`**: Linear $\alpha$ ensemble metrics.